# Graph Neural Networks from Scratch — Tutorial 6: Graph Attention Networks (GAT)

### From equal neighbour aggregation to learning which neighbours matter

*📺 Video on YouTube | 🐍 Python · NumPy · PyTorch · PyTorch Geometric | Beginner–Intermediate*

*Notebook repository: [github.com/rraj-io/GNN-Tutorials](https://github.com/rraj-io/GNN-Tutorials)*

---

## Series Roadmap

In Tutorial 1, we learned how graphs represent connected data through nodes, edges, node features, and adjacency matrices. In Tutorial 2, we implemented graph convolution from scratch using NumPy. In Tutorial 3, we made a GCN trainable with PyTorch. We then moved to PyTorch Geometric and the Cora citation network, explored message passing, and studied GraphSAGE-style aggregation.

So far, many of our aggregation methods have treated neighbours equally or used a fixed normalization rule. But should every neighbour always contribute equally?

Imagine a scientific paper connected to several other papers. Some neighbouring papers may be highly relevant to its topic, while others may provide less useful information. A graph model should ideally learn these differences.

This is the motivation behind **Graph Attention Networks (GATs)**. In this tutorial, we will develop attention step by step, implement a single-head GAT layer using PyTorch, understand multi-head attention, and then use PyTorch Geometric's `GATConv` on Cora.

| # | Tutorial | Focus | Status |
|---|---|---|---|
| 1 | Graph Fundamentals and Representations | nodes, edges, adjacency matrices, graph features | ✅ |
| 2 | Graph Convolutional Networks from Scratch | propagation, normalization, NumPy implementation | ✅ |
| 3 | Training a GCN with PyTorch | trainable parameters, backpropagation, node classification | ✅ |
| 4 | PyTorch Geometric and the Cora Dataset | practical GNN implementation, real graph dataset | ✅ |
| 5 | Message Passing Neural Networks | message, aggregate, update, custom GNN layers | ✅ |
| **6** | **Graph Attention Networks (GAT)** | **learned attention, neighbour weighting, multi-head attention** | **📍 You are here** |
| 7 | GraphSAGE and Advanced Aggregation | scalable neighbourhood learning | Planned |
| 8 | Graph-Level Learning | graph classification and regression | Planned |
| 9 | GNNs for Mesh-Based Systems | mesh connectivity, geometric features, scientific data | Planned |
| 10 | GNNs for Scientific Machine Learning | surrogate modelling and engineering applications | Planned |

> **Prerequisites:** This tutorial builds on the GCN and message-passing concepts from Tutorials 2–5. You should be comfortable with basic Python, PyTorch tensors, neural-network training, and sending messages along graph edges. We will recap the key equations as we go.

## What will we learn?

- Why equal neighbour weighting can be limiting.
- How a node computes an attention score for each neighbour.
- How softmax turns scores into normalized attention coefficients.
- How attention-weighted messages are aggregated.
- What multi-head attention does and why it is useful.
- How to implement a basic GAT layer with PyTorch.
- How to use `GATConv` for node classification on Cora.
- How to inspect attention coefficients—and why they should be interpreted carefully.

## What we will build

```text
Node Features
     |
     v
Linear Feature Projection
     |
     v
Compute Neighbour Attention Scores
     |
     v
Neighbourhood Softmax
     |
     v
Weighted Message Aggregation
     |
     v
Multi-Head Attention
     |
     v
Updated Node Representations
     |
     v
Node Classification
```

**Let's teach our graph network not only to listen to its neighbours, but also to learn how much attention to give each one.**

## 1. Why do we need graph attention?

A simple mean aggregator gives each neighbour the same coefficient:

\[
\mathbf{h}'_i = \frac{1}{|\mathcal{N}(i)|}
\sum_{j\in\mathcal{N}(i)} \mathbf{h}_j
\]

This is useful, but it assumes all neighbours are equally informative. A GAT replaces fixed weighting with learned coefficients:

\[
\mathbf{h}'_i =
\sigma\left(
\sum_{j\in\mathcal{N}(i)\cup\{i\}}
\alpha_{ij}\mathbf{W}\mathbf{h}_j
\right)
\]

Here, \(\alpha_{ij}\) measures how strongly source node \(j\) contributes when updating destination node \(i\).

**Important:** attention does not change which nodes are connected. The graph determines the allowed neighbourhood; attention learns how to weight information arriving along those connections.

## 2. The GAT attention mechanism

**Step 1 — Linear feature transformation**

\[
\mathbf{z}_i = \mathbf{W}\mathbf{h}_i
\]

Here, \(\mathbf{h}_i\) is the input feature vector, \(\mathbf{W}\) is a trainable projection matrix, and \(\mathbf{z}_i\) is the transformed representation.

**Step 2 — Compute an unnormalized attention score**

For a message from source \(j\) to destination \(i\):

\[
e_{ij} =
\operatorname{LeakyReLU}
\left(
\mathbf{a}^{T}
[\mathbf{z}_i \Vert \mathbf{z}_j]
\right)
\]

The trainable attention vector \(\mathbf{a}\) scores the compatibility of the two transformed node representations.

**Step 3 — Normalize over the destination's neighbourhood**

\[
\alpha_{ij} =
\frac{\exp(e_{ij})}
{\sum_{k\in\mathcal{N}(i)\cup\{i\}}\exp(e_{ik})}
\]

For each destination node, the incoming attention coefficients sum to one:

\[
\sum_{j\in\mathcal{N}(i)\cup\{i\}}\alpha_{ij}=1
\]

**Step 4 — Aggregate weighted messages**

\[
\mathbf{h}'_i =
\sigma\left(
\sum_{j\in\mathcal{N}(i)\cup\{i\}}
\alpha_{ij}\mathbf{z}_j
\right)
\]

This is the central operation of a single-head GAT layer.

**Self-loops:** the node itself is included in its attention neighbourhood, allowing it to use its own transformed features as well as those of its neighbours.

## 3. A tiny numerical attention example

Suppose a destination node receives three unnormalized scores:

\[
e_{i1}=0.0,\qquad e_{i2}=1.0,\qquad e_{i3}=2.0
\]

Softmax assigns larger weights to larger scores while ensuring that the weights sum to one.

In [ ]:
import numpy as np

scores = np.array([0.0, 1.0, 2.0])

# Numerically stable softmax
exp_scores = np.exp(scores - scores.max())
attention = exp_scores / exp_scores.sum()

print("Scores:", scores)
print("Attention coefficients:", attention)
print("Sum:", attention.sum())

The largest score receives the largest attention coefficient. These coefficients are relative to the other scores in the same destination node's neighbourhood. They are not universal importance scores.

**Try it:** change the scores. What happens when all scores are equal?

## 4. Graph direction and edge indexing

We represent edges with a PyTorch `edge_index` tensor of shape `[2, number_of_edges]`.

For an edge \(j\rightarrow i\):
- `edge_index[0]` stores the source node \(j\).
- `edge_index[1]` stores the destination node \(i\).

The message is sent from source to destination. Attention normalization must therefore group scores by **destination node**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

# Directed edges. Self-loops are added inside the GAT layer.
edge_index = torch.tensor([
    [0, 1, 2, 0, 2, 3],
    [1, 0, 1, 2, 3, 2]
], dtype=torch.long)

num_nodes = 4
x = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.2],
    [0.5, 0.5, 1.0],
    [1.0, 1.0, 0.0]
])

print("Node features:", x.shape)
print("Edge index:", edge_index.shape)
print("Sources:", edge_index[0])
print("Destinations:", edge_index[1])

## 5. Implementing destination-wise softmax

In ordinary softmax, all entries belong to one group. In graph attention, each destination node has its own group of incoming edge scores.

For numerical stability, subtract the maximum score within each destination group before exponentiation. The implementation below uses PyTorch's `scatter_reduce_` and `index_add_`. It is intended for learning and small examples; optimized graph libraries provide more efficient implementations.

In [ ]:
def destination_softmax(scores, destination, num_nodes):
    """Softmax edge scores separately for each destination node.

    scores:      [num_edges, num_heads]
    destination: [num_edges]
    returns:     [num_edges, num_heads]
    """
    num_heads = scores.size(1)
    max_per_destination = scores.new_full(
        (num_nodes, num_heads), -torch.inf
    )

    max_per_destination.scatter_reduce_(
        0,
        destination[:, None].expand(-1, num_heads),
        scores,
        reduce="amax",
        include_self=True
    )

    exp_scores = (scores - max_per_destination[destination]).exp()
    denominator = scores.new_zeros((num_nodes, num_heads))
    denominator.index_add_(0, destination, exp_scores)

    return exp_scores / denominator[destination].clamp_min(1e-16)

In [ ]:
demo_scores = torch.tensor([[0.0], [1.0], [2.0], [0.5], [1.5], [0.2]])
demo_destinations = torch.tensor([0, 0, 0, 1, 1, 2])

demo_alpha = destination_softmax(
    demo_scores, demo_destinations, num_nodes=3
)

for node_id in range(3):
    mask = demo_destinations == node_id
    print(
        f"Destination {node_id}: "
        f"coefficients={demo_alpha[mask, 0].tolist()}, "
        f"sum={demo_alpha[mask].sum().item():.4f}"
    )

## 6. Build a single-head GAT layer from scratch

We combine the operations:

1. Project node features with a trainable matrix.
2. Compute a score for every directed edge.
3. Apply LeakyReLU.
4. Normalize scores by destination using grouped softmax.
5. Weight source features by attention.
6. Sum messages at each destination.
7. Concatenate or average the attention heads.

For clarity, this educational implementation uses explicit tensor indexing and `index_add_`.

In [ ]:
class GATLayer(nn.Module):
    def __init__(
        self, in_features, out_features, num_heads=1,
        concat=True, dropout=0.0, negative_slope=0.2,
        add_self_loops=True
    ):
        super().__init__()
        self.out_features = out_features
        self.num_heads = num_heads
        self.concat = concat
        self.dropout = dropout
        self.negative_slope = negative_slope
        self.add_self_loops = add_self_loops

        self.lin = nn.Linear(
            in_features, num_heads * out_features, bias=False
        )
        self.att_src = nn.Parameter(torch.empty(1, num_heads, out_features))
        self.att_dst = nn.Parameter(torch.empty(1, num_heads, out_features))
        output_dim = num_heads * out_features if concat else out_features
        self.bias = nn.Parameter(torch.zeros(output_dim))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, return_attention=False):
        num_nodes = x.size(0)
        source, destination = edge_index

        if self.add_self_loops:
            nodes = torch.arange(num_nodes, device=x.device)
            loops = torch.stack([nodes, nodes], dim=0)
            edge_index = torch.cat([edge_index, loops], dim=1)
            source, destination = edge_index

        # [num_nodes, num_heads, out_features]
        z = self.lin(x).view(
            num_nodes, self.num_heads, self.out_features
        )

        # Additive attention score for each source -> destination edge.
        score_src = (z * self.att_src).sum(dim=-1)
        score_dst = (z * self.att_dst).sum(dim=-1)
        edge_scores = score_src[source] + score_dst[destination]
        edge_scores = F.leaky_relu(
            edge_scores, negative_slope=self.negative_slope
        )

        alpha = destination_softmax(edge_scores, destination, num_nodes)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        messages = z[source] * alpha.unsqueeze(-1)
        out = z.new_zeros((num_nodes, self.num_heads, self.out_features))
        out.index_add_(0, destination, messages)

        if self.concat:
            out = out.reshape(num_nodes, self.num_heads * self.out_features)
        else:
            out = out.mean(dim=1)

        out = out + self.bias
        if return_attention:
            return out, (edge_index, alpha)
        return out

In [ ]:
gat = GATLayer(in_features=3, out_features=4, num_heads=1)
out, (edges_with_loops, alpha) = gat(x, edge_index, return_attention=True)

print("Output shape:", out.shape)
print("Edges including self-loops:", edges_with_loops.shape)
print("Attention shape:", alpha.shape)
print(out)

### Inspect the attention coefficients

Set the layer to evaluation mode so attention dropout is disabled. The incoming coefficients should sum to one for each destination node.

In [ ]:
gat.eval()
with torch.no_grad():
    _, (edges_with_loops, alpha) = gat(
        x, edge_index, return_attention=True
    )

src, dst = edges_with_loops
for node_id in range(num_nodes):
    incoming = dst == node_id
    print(
        f"Node {node_id}: sources={src[incoming].tolist()}, "
        f"weights={alpha[incoming, 0].tolist()}, "
        f"sum={alpha[incoming, 0].sum().item():.4f}"
    )

## 7. Multi-head attention

A single attention head may focus on one pattern in a neighbourhood. GAT uses multiple heads to learn several attention patterns in parallel.

For \(K\) heads, each head computes its own attention coefficients and aggregated representation. In hidden layers, head outputs are commonly concatenated:

\[
\mathbf{h}'_i =
\mathop{\Vert}_{k=1}^{K}
\sigma\left(
\sum_{j\in\mathcal{N}(i)\cup\{i\}}
\alpha_{ij}^{(k)}\mathbf{W}^{(k)}\mathbf{h}_j
\right)
\]

In a final prediction layer, head outputs are often averaged instead.

In [ ]:
multi_concat = GATLayer(3, 4, num_heads=3, concat=True)
multi_average = GATLayer(3, 4, num_heads=3, concat=False)

print("Concatenated heads:", multi_concat(x, edge_index).shape)
print("Averaged heads:", multi_average(x, edge_index).shape)

With three heads and four features per head:
- Concatenation produces 12 features per node.
- Averaging produces 4 features per node.

More heads are not automatically better. They increase computation and model capacity.

## 8. Train a GAT on the Cora citation network with PyTorch Geometric

We have implemented the central attention operation ourselves. Now we will use PyTorch Geometric's optimized `GATConv` layer to train a model on Cora.

Cora is a citation network:
- Nodes represent scientific publications.
- Edges represent citation links.
- Node features describe the publications.
- Labels represent publication categories.

This is a transductive node-classification task: graph structure and node features are available during training, while the training mask determines which labels contribute to the loss.

### Installation

Install a PyTorch version compatible with your system, then follow the official PyG instructions for the matching PyTorch and CUDA versions.

```bash
pip install torch
pip install torch-geometric
pip install matplotlib
```

- [PyTorch installation](https://pytorch.org/get-started/locally/)
- [PyTorch Geometric installation](https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html)

In [ ]:
import matplotlib.pyplot as plt
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

dataset = Planetoid(root="data/Planetoid", name="Cora")
data = dataset[0].to(device)

print(dataset)
print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)
print("Input features:", dataset.num_features)
print("Classes:", dataset.num_classes)
print("Train nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

## 9. Define a two-layer GAT model

The first layer uses multiple attention heads and concatenates their outputs. The second layer uses one head and produces one logit per class.

We use dropout as regularization. `GATConv` adds self-loops by default.

In [ ]:
class GAT(nn.Module):
    def __init__(
        self, in_channels, hidden_channels, out_channels,
        heads=8, dropout=0.6
    ):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(
            in_channels, hidden_channels,
            heads=heads, concat=True, dropout=dropout
        )
        self.conv2 = GATConv(
            hidden_channels * heads, out_channels,
            heads=1, concat=False, dropout=dropout
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

## 10. Training and evaluation

The model returns **logits**, not probabilities. `CrossEntropyLoss` expects raw logits and internally applies the appropriate log-softmax operation.

We train only on nodes selected by `train_mask`, monitor `val_mask`, and reserve `test_mask` for final evaluation.

In [ ]:
model = GAT(
    in_channels=dataset.num_features,
    hidden_channels=8,
    out_channels=dataset.num_classes,
    heads=8,
    dropout=0.6
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.005, weight_decay=0.0005
)
criterion = nn.CrossEntropyLoss()
print(model)

In [ ]:
@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    predictions = logits[mask].argmax(dim=1)
    accuracy = (predictions == data.y[mask]).float().mean().item()
    loss = criterion(logits[mask], data.y[mask]).item()
    return loss, accuracy


def train_one_epoch():
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
num_epochs = 200
history = {
    "train_loss": [], "train_accuracy": [],
    "val_loss": [], "val_accuracy": []
}

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch()
    _, train_acc = evaluate(model, data, data.train_mask)
    val_loss, val_acc = evaluate(model, data, data.val_mask)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | Train loss: {train_loss:.4f} | "
            f"Train acc: {train_acc:.4f} | Val loss: {val_loss:.4f} | "
            f"Val acc: {val_acc:.4f}"
        )

## 11. Visualize the training process

In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"], label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["train_accuracy"], label="Train")
axes[1].plot(epochs, history["val_accuracy"], label="Validation")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Final test-set evaluation

For a more rigorous experiment, save the checkpoint with the best validation score and evaluate that checkpoint once on the test set. Here, we report the final-epoch result for simplicity.

In [ ]:
test_loss, test_accuracy = evaluate(model, data, data.test_mask)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 13. Inspect attention weights with `GATConv`

PyG can return the attention coefficients computed by a GAT layer. The returned edge list may include self-loops, because `GATConv` adds them by default.

Attention weights should be interpreted carefully:
- They depend on the model, layer, head, and current input.
- They are normalized within a destination node's neighbourhood.
- A large coefficient does not automatically mean a feature is causally or physically important.
- Attention is not a substitute for a full model-explanation or uncertainty analysis.

In [ ]:
model.eval()
with torch.no_grad():
    _, (attention_edge_index, attention_weights) = model.conv1(
        data.x,
        data.edge_index,
        return_attention_weights=True
    )

print("Returned edge index:", attention_edge_index.shape)
print("Attention weights:", attention_weights.shape)
print("Number of heads:", attention_weights.size(1))
print("First few coefficients:")
print(attention_weights[:10])

In [ ]:
destination_node = int(data.edge_index[1, 0].item())
head_id = 0

src_nodes = attention_edge_index[0]
dst_nodes = attention_edge_index[1]
incoming_mask = dst_nodes == destination_node

incoming_sources = src_nodes[incoming_mask]
incoming_alpha = attention_weights[incoming_mask, head_id]

print("Destination node:", destination_node)
for source_node, alpha_value in zip(
    incoming_sources.tolist(), incoming_alpha.tolist()
):
    print(
        f"Source {source_node:5d} -> Destination {destination_node:5d}: "
        f"alpha = {alpha_value:.4f}"
    )
print("Sum:", incoming_alpha.sum().item())

## 14. GCN versus GAT: what is different?

| Property | GCN | GAT |
|---|---|---|
| Neighbour weighting | Fixed graph normalization | Learned attention coefficients |
| Feature transformation | Trainable | Trainable |
| Neighbour contributions | Determined by normalized adjacency | Adaptively weighted by features |
| Multiple heads | Not part of standard GCN | Supported |
| Main intuition | Combine neighbourhood information using graph structure | Learn which neighbours to emphasize |

Neither architecture is universally better. GAT can learn flexible, feature-dependent weighting, but attention adds parameters and computation. A GCN can be simpler and more efficient, and may perform very well when fixed normalized aggregation is appropriate.

## 15. Important limitations and practical considerations

1. **Attention is not automatically explainability.** Attention coefficients show how a layer weights messages, but do not by themselves establish causal importance.
2. **Attention is local.** A node's coefficients are normalized over its own incoming neighbourhood; coefficients from different nodes are not directly comparable.
3. **Compute cost grows with heads.** Multi-head attention can increase memory use and runtime.
4. **Neighbourhood size matters.** Nodes with many neighbours require more edge-level attention calculations.
5. **Edge information is not automatically used.** The basic GAT formulation here computes attention from node features. If edge attributes matter, the architecture must be extended.
6. **Graph direction matters.** Citation datasets may be treated as directed or symmetrized depending on preprocessing. Be explicit about the convention used.
7. **Attention does not enforce physical laws.** A GAT does not automatically conserve mass, momentum, or energy. Such constraints must be built into the model, objective, or message-passing design when required.

## 16. Connecting GATs to computational meshes and physical systems

A computational mesh can be represented as a graph:
- Mesh vertices or cell centres become graph nodes.
- Mesh connectivity becomes graph edges.
- Pressure, velocity, temperature, and material properties become node features.
- Geometric quantities such as distance, face area, orientation, and volume can be stored as edge or node attributes.

A GAT could learn to assign different weights to neighbouring cells when constructing a local representation. This may be useful when some neighbours provide more informative context than others.

However, a standard GAT is not a finite-volume discretization. Its learned attention coefficients are not automatically face fluxes, interpolation weights, or conservative numerical operators.

For engineering applications, useful next steps include incorporating relative positions and edge attributes, comparing attention with geometry-aware message passing, enforcing conservation and boundary conditions, and evaluating generalization across mesh resolutions and geometries.

## 17. Hands-on exercises

### Exercise 1 — Change the number of attention heads
Train with 1, 2, 4, and 8 heads. Keep the hidden dimension per head fixed. Compare validation accuracy, parameter count, and training time.

### Exercise 2 — Change the hidden dimension
Try `hidden_channels=4`, `8`, and `16`. Does increasing model capacity consistently improve validation performance?

### Exercise 3 — Remove attention dropout
Try dropout values of `0.0`, `0.3`, and `0.6`. Compare training and validation curves. Look for signs of overfitting.

### Exercise 4 — Compare GCN and GAT
Use the same Cora split and training protocol to compare a GCN and a GAT. Do not compare results obtained with different data splits without noting that difference.

### Exercise 5 — Inspect multiple heads
For one destination node, compare attention coefficients from different heads. Do the heads distribute attention similarly, or focus on different neighbours?

### Exercise 6 — Add edge information (conceptual)
Suppose each edge has a distance and a relative direction. Sketch how these quantities could be incorporated into the attention score. You do not need to implement the extension yet.

## 18. Final challenge — Build a controlled GCN versus GAT comparison

Create a small experimental study on Cora:

1. Train a GCN baseline.
2. Train a GAT with one attention head.
3. Train a GAT with multiple attention heads.
4. Use the same train, validation, and test masks for every model.
5. Keep the evaluation protocol consistent.
6. Record validation accuracy, test accuracy, parameter count, and approximate training time.
7. Discuss the trade-off between model complexity and predictive performance.

Your comparison should answer:
- Does learned neighbour weighting improve performance on this dataset?
- Do more attention heads help?
- Is the performance gain worth the additional computation?
- Are learned attention patterns stable across heads?

Do not assume GAT must outperform GCN. A careful comparison is more valuable than a result chosen to favour one architecture.

## 19. Summary

In this tutorial, we learned how Graph Attention Networks extend neighbourhood aggregation.

We covered:
- Why fixed or equal neighbour weighting can be limiting.
- How to project node features into a learned representation space.
- How to compute additive attention scores.
- How neighbourhood softmax produces normalized attention coefficients.
- How to aggregate weighted messages.
- How multi-head attention learns several patterns in parallel.
- How to implement a basic GAT layer with PyTorch.
- How to train a GAT using PyTorch Geometric on Cora.
- Why attention weights must be interpreted carefully.

The key idea is that **the graph determines which neighbours can communicate, while attention learns how strongly their messages contribute to each node's representation.**

## What's next?

In Tutorial 7, we will explore **GraphSAGE and Advanced Aggregation**. We will focus on scalable neighbourhood sampling, different aggregation strategies, and how GNNs can learn representations for larger graphs where processing every node and edge at once becomes expensive.

From there, we will move towards graph-level learning and, eventually, GNNs for computational meshes and scientific machine learning.

---

**Learn the graph. Understand the message passing. Build the model. Connect it to the physical world.**